In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from apis import ticketmaster as tm

In [ ]:
import apis.events.client_models.ticketmaster_event_model as em
import apis.events.event_record as er
import apis.events.events_mapper as ema

In [ ]:
endpoint = "/events.json"
embedded_parts = [("_embedded", "events")]
kword_filters_gb= {"keyword":"festival", "countryCode":"GB"}
kword_filters_uk= {"keyword":"festival", "countryCode":"UK"}
events_list = []

response_data_gb = tm.get_ticketmaster_data(endpoint, embedded_parts, filters=kword_filters_gb)
response_data_uk = tm.get_ticketmaster_data(endpoint, embedded_parts, filters=kword_filters_uk)
print(response_data_gb)
events_list = response_data_gb + response_data_uk

In [ ]:
endpoint = "/venues.json"
embedded_parts = [("_embedded", "venues")]
kword_filters_gb= {"countryCode":"GB"}
kword_filters_uk= {"countryCode":"UK"}
venues_list = []

response_data_gb = tm.get_ticketmaster_data(endpoint, embedded_parts, filters=kword_filters_gb)
response_data_uk = tm.get_ticketmaster_data(endpoint, embedded_parts, filters=kword_filters_uk)
venues_list = response_data_gb + response_data_uk

In [ ]:
from data_processor.ticketmaster.venues_processor import VenuesProcessor

venues_models, invalid_venue_models = VenuesProcessor.validate_venues(venues_list)
venues_df = VenuesProcessor.build_venues_dataframe(venues_models)

In [ ]:
from data_processor.ticketmaster.events_processor import EventsProcessor

events_models, invalid_event_models = EventsProcessor.validate_events(events_list)
events_df = EventsProcessor.build_events_dataframe(events_models)

In [ ]:
from data_processor.ticketmaster.ticketmaster_processor import TicketmasterProcessor

evp = EventsProcessor()
vnp = VenuesProcessor()

tmp = TicketmasterProcessor(events_df, venues_df, evp, vnp)
merged_df = tmp.merge_venues_to_events()

merged_df.head()